# Clase Plotly interactiva para explorar ventas y movimientos de almacén


## Propósito de negocio

Explorar dos perspectivas operativas relacionadas, pero distintas:

- **Ventas:** unidades vendidas y venta neta registradas por artículo.
- **Movimientos de almacén:** unidades de salida registradas por SKU y mes.





## 1. Objetivos de aprendizaje



1. Preparar datos para visualización: normalizar llaves, construir periodos, agregar y validar granularidad.
2. Explicar por qué se agrega antes de unir (`groupby` + `merge`).
3. Crear visualizaciones con Plotly Express: `bar`, `line`, `scatter`, `treemap`.
4. Añadir interactividad útil: hover, filtros, selección de marca y rango temporal.
5. Comunicar hallazgos sin atribuir causalidad ni convertir señales exploratorias en decisiones de compra.



## 2. Instalación e importaciones

Si es necesario, ejecuta la instalación. En Jupyter puede requerirse reiniciar el kernel después de instalar paquetes.

In [ ]:
%pip install -q pandas numpy plotly openpyxl ipywidgets

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")



MAIN_FILE = Path.cwd() / 'Datos' / 'Datos_CLV_2026_Tec_v2.xlsx'
COMP_FILE = Path.cwd() / 'Datos' / 'Datos_Complementarios_CLV.xlsx'
MAIN_FILE = Path(MAIN_FILE)
COMP_FILE = Path(COMP_FILE)
print("Libro principal:", MAIN_FILE.resolve())
print("Libro complementario:", COMP_FILE.resolve())

## 3. Carga de hojas y revisión del esquema

Las hojas se buscan por nombres candidatos para tolerar pequeñas diferencias de escritura. No se renombra una columna de forma silenciosa: primero se inspecciona el esquema.

In [ ]:
principal = pd.read_excel(MAIN_FILE, sheet_name=None)
complementario = pd.read_excel(COMP_FILE, sheet_name=None)

print("Hojas del libro principal:")
print(list(principal))
print("\nHojas del libro complementario:")
print(list(complementario))

def obtener_hoja(libro, candidatos):
    for nombre in candidatos:
        if nombre in libro:
            return libro[nombre].copy()
    raise KeyError(
        f"No se encontró ninguna hoja entre {candidatos}. "
        f"Hojas disponibles: {list(libro)}"
    )

ventas = obtener_hoja(
    principal,
    ["Ventas_2022_2026", "Ventas_CLV_2022_2026", "Ventas_2022_2026 "]
)
mov = obtener_hoja(
    complementario,
    ["Histórico_Entradas_SAlidas", "Historico_Entradas_SAlidas"]
)

print("Ventas:", ventas.shape)
print("Movimientos:", mov.shape)
display(ventas.head(3))
display(mov.head(3))

### Preguntas de comprobación

- ¿Qué representa una fila en ventas? Una línea/documento de venta, no necesariamente un SKU único.
- ¿Qué representa una fila en movimientos? Un registro por combinación de periodo, SKU y posiblemente almacén; la granularidad se confirma con las columnas.
- ¿Por qué no graficar directamente las tablas crudas? Porque las repeticiones pueden sobrerrepresentar productos y la visualización podría sumar o contar unidades en un nivel inadecuado.

## 4. Preparación: llaves, fechas y tipos

Normalizar una llave elimina diferencias de formato (espacios, mayúsculas, sufijo `.0`). **No prueba** que dos sistemas utilicen la misma codificación semántica.

In [ ]:
def normalizar_llave(serie):
    s = serie.astype("string").str.strip().str.upper()
    s = s.str.replace(r"\.0$", "", regex=True)
    return s.replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA})

# Verificación de columnas mínimas
cols_ventas = {"clave_articulo", "fecha_documento", "total_unidades", "total_neto"}
cols_mov = {"SKU", "ANO", "MES", "SALIDAS"}
faltan_v = cols_ventas - set(ventas.columns)
faltan_m = cols_mov - set(mov.columns)
if faltan_v or faltan_m:
    raise ValueError(f"Faltan columnas. Ventas: {faltan_v}; movimientos: {faltan_m}")

ventas["clave_key"] = normalizar_llave(ventas["clave_articulo"])
ventas["fecha_documento"] = pd.to_datetime(ventas["fecha_documento"], errors="coerce")
ventas["total_unidades"] = pd.to_numeric(ventas["total_unidades"], errors="coerce")
ventas["total_neto"] = pd.to_numeric(ventas["total_neto"], errors="coerce")

mov["sku_key"] = normalizar_llave(mov["SKU"])
mov["ANO"] = pd.to_numeric(mov["ANO"], errors="coerce")
mov["MES"] = pd.to_numeric(mov["MES"], errors="coerce")
mov["SALIDAS"] = pd.to_numeric(mov["SALIDAS"], errors="coerce")
mov["periodo"] = pd.to_datetime(
    dict(year=mov["ANO"], month=mov["MES"], day=1), errors="coerce"
)

print("Fechas válidas ventas:", f"{ventas['fecha_documento'].notna().mean():.1%}")
print("Periodos válidos movimientos:", f"{mov['periodo'].notna().mean():.1%}")
print("Llaves válidas ventas:", f"{ventas['clave_key'].notna().mean():.1%}")
print("Llaves válidas movimientos:", f"{mov['sku_key'].notna().mean():.1%}")
ventas.head(3)
mov.head(3)

### Interpretación 

El periodo de movimientos se construye con `ANO` y `MES`, que son los campos temporales disponibles y más consistentes para esta fuente. No se usa la fecha de `Salidas_Almacen` del libro principal: en la auditoría previa presentó una proporción elevada de fechas no interpretables.

Si la proporción de fechas o llaves válidas cambia sustancialmente, se debe detener la interpretación y revisar la extracción.

## 5. Definir una ventana temporal común

Para comparar ventas y salidas por SKU se utiliza una ventana común de 12 meses calendario, definida con base en el último periodo válido de movimientos. La ventana común reduce el sesgo de comparar periodos diferentes.

**Importante:** un mes sin registros no necesariamente equivale a cero demanda o cero salidas. En este ejercicio sumamos registros observados; no fabricamos ceros para meses ausentes.

In [ ]:
mov_ok = mov.dropna(subset=["sku_key", "periodo", "SALIDAS"]).copy()
ventas_ok = ventas.dropna(subset=["clave_key", "fecha_documento", "total_unidades", "total_neto"]).copy()

periodo_max_mov = mov_ok["periodo"].max()
if pd.isna(periodo_max_mov):
    raise ValueError("No hay periodos válidos en movimientos.")
periodo_min = periodo_max_mov - pd.DateOffset(months=11)

mov_12m = mov_ok.loc[mov_ok["periodo"].between(periodo_min, periodo_max_mov)].copy()
ventas_ok["periodo"] = ventas_ok["fecha_documento"].dt.to_period("M").dt.to_timestamp()
ventas_12m = ventas_ok.loc[ventas_ok["periodo"].between(periodo_min, periodo_max_mov)].copy()

print(f"Ventana común: {periodo_min:%Y-%m} a {periodo_max_mov:%Y-%m}")
print("Registros movimientos:", len(mov_12m))
print("Registros ventas:", len(ventas_12m))

In [ ]:
# Agregar movimientos a una fila por SKU.
mov_sku = (
    mov_12m.groupby("sku_key", as_index=False)
    .agg(
        salidas_12m=("SALIDAS", "sum"),
        meses_con_movimiento=("periodo", "nunique"),
    )
)

# Agregar ventas a una fila por clave de artículo.
ventas_sku = (
    ventas_12m.groupby("clave_key", as_index=False)
    .agg(
        unidades_vendidas_12m=("total_unidades", "sum"),
        venta_neta_12m=("total_neto", "sum"),
        clientes_12m=("nombre_cliente", "nunique") if "nombre_cliente" in ventas_12m.columns else ("clave_key", "size"),
        meses_con_venta=("periodo", "nunique")
    )
)

print("SKU únicos en movimientos:", mov_sku["sku_key"].nunique())
print("Claves únicas en ventas:", ventas_sku["clave_key"].nunique())
print("¿Una fila por llave en movimientos?", mov_sku["sku_key"].is_unique)
print("¿Una fila por llave en ventas?", ventas_sku["clave_key"].is_unique)
display(mov_sku.head())
display(ventas_sku.head())

### ¿Por qué agregar antes del merge?

Las dos fuentes contienen varias filas por artículo. Unirlas directamente puede producir una relación muchos-a-muchos: una venta se replica tantas veces como movimientos coincidentes, inflando totales. Al agregar cada fuente primero, se obtiene una fila por llave y el `merge` puede validarse como uno-a-uno.

## 6. Unión y cobertura de llaves

Se usa `outer` para conservar tanto artículos presentes en ventas como SKU presentes en movimientos. El indicador `_merge` permite distinguir coincidencias y registros exclusivos.

In [ ]:
analisis = ventas_sku.merge(
    mov_sku,
    left_on="clave_key",
    right_on="sku_key",
    how="outer",
    indicator=True,
    validate="one_to_one"
)

conteo_union = analisis["_merge"].value_counts().reindex(
    ["both", "left_only", "right_only"], fill_value=0
)
resumen_union = pd.DataFrame({
    "SKU/llaves": conteo_union,
    "Porcentaje del universo unido": (conteo_union / len(analisis) * 100).round(2)
})
display(resumen_union)

base_mov = len(mov_sku)
match_mov = int((analisis["_merge"] == "both").sum())
cobertura_mov = 100 * match_mov / base_mov if base_mov else np.nan
print(f"De los SKU con movimientos, {cobertura_mov:.1f}% tienen correspondencia en ventas.")

### Interpretación: leer correctamente el porcentaje

El porcentaje calculado usa como denominador los SKU únicos con movimientos en la ventana. Responde: **¿qué proporción de SKU con movimientos aparece también en ventas?**

No responde:
- qué porcentaje de unidades de salida está cubierto;
- qué porcentaje de ventas está cubierto;
- qué porcentaje del catálogo completo está cubierto;
- qué proporción de la demanda fue satisfecha.

`left_only` representa una clave de ventas sin coincidencia en movimientos; `right_only` representa un SKU con movimientos sin coincidencia en ventas. Ninguno debe convertirse automáticamente en cero ni eliminarse sin investigar.

## 7. Plotly 1 · tendencia mensual de ventas y salidas

Creamos dos figuras separadas. Aunque ambas son cantidades, las definiciones y procesos son diferentes. No las superponemos ni sumamos.

In [ ]:
ventas_mes = (
    ventas_12m.groupby("periodo", as_index=False)
    .agg(unidades_vendidas=("total_unidades", "sum"))
)
salidas_mes = (
    mov_12m.groupby("periodo", as_index=False)
    .agg(unidades_salida=("SALIDAS", "sum"))
)

fig_ventas = px.line(
    ventas_mes, x="periodo", y="unidades_vendidas", markers=True,
    title="Unidades vendidas por mes · ventana común",
    labels={"periodo": "Mes", "unidades_vendidas": "Unidades vendidas"},
    template="plotly_white"
)
fig_ventas.update_layout(hovermode="x unified")
fig_ventas.update_xaxes(rangeslider_visible=True)
fig_ventas.show()

fig_salidas = px.line(
    salidas_mes, x="periodo", y="unidades_salida", markers=True,
    title="Unidades de salida registradas por mes · ventana común",
    labels={"periodo": "Mes", "unidades_salida": "Unidades de salida"},
    template="plotly_white"
)
fig_salidas.update_layout(hovermode="x unified")
fig_salidas.update_xaxes(rangeslider_visible=True)
fig_salidas.show()

### Cómo interpretar estas gráficas

- Cada punto representa el total registrado en un mes.
- El *range slider* permite acercarse a un intervalo sin recalcular la figura.
- El hover unificado facilita leer los valores por fecha.
- Buscar picos, caídas, meses sin observaciones y cambios de patrón.
- No atribuir un pico a una campaña, temporada o problema de inventario sin evidencia adicional.

**Pregunta:** si ventas suben y salidas bajan, ¿podemos concluir que el almacén dejó de surtir?  
**No**. Puede haber diferencias de corte, reconocimiento, devoluciones, movimientos internos, unidades, retrasos de captura o una relación temporal distinta. Se requiere investigar.

## 8. Plotly 2 · dispersión ventas vs salidas por SKU

Cada punto representa una llave coincidente. Los ejes mantienen las dos medidas separadas. El color puede ayudar a explorar grupos; si no existe una marca confiable en la tabla agregada, no se inventa.

In [ ]:
coincidentes = analisis.loc[analisis["_merge"].eq("both")].copy()
if "MARCA" in mov_12m.columns:
    marca_sku = (mov_12m.groupby("sku_key", as_index=False)
                 .agg(marca=("MARCA", lambda x: x.dropna().mode().iat[0] if not x.dropna().mode().empty else "Sin marca")))
    coincidentes = coincidentes.merge(marca_sku, on="sku_key", how="left", validate="one_to_one")
else:
    coincidentes["marca"] = "No disponible"

fig_scatter = px.scatter(
    coincidentes,
    x="unidades_vendidas_12m",
    y="salidas_12m",
    color="marca",
    hover_name="sku_key",
    hover_data={
        "venta_neta_12m": ":,.2f",
        "clientes_12m": True,
        "meses_con_venta": True,
        "meses_con_movimiento": True,
        "sku_key": True
    },
    title="Ventas registradas vs salidas registradas por SKU",
    labels={
        "unidades_vendidas_12m": "Unidades vendidas (12 meses)",
        "salidas_12m": "Unidades de salida (12 meses)",
        "marca": "Marca"
    },
    opacity=0.7,
    template="plotly_white"
)
fig_scatter.update_layout(legend_title_text="Marca")
fig_scatter.show()

### Interpretación de la dispersión

- Un punto representa un SKU, no una transacción.
- Puntos arriba de la diagonal implícita tienen más salidas registradas que unidades vendidas; puntos debajo tienen más unidades vendidas que salidas.
- La distancia respecto a la diagonal es una señal para investigar definiciones, periodos, unidades y procesos, no una prueba de error.
- Un punto aislado puede ser un SKU de alto volumen o un caso de captura que merece revisión.
- Hover permite consultar el SKU y métricas auxiliares sin saturar la figura.

**No interpretar correlación como equivalencia ni causalidad.** La venta neta es monetaria y no debe usarse como tamaño del punto para representar rentabilidad.

## 9. Plotly 3 · Top 15 SKU por salidas

La barra horizontal facilita leer claves largas. El Top 15 se construye después de agregar por SKU.

In [ ]:
top_salidas = mov_sku.nlargest(15, "salidas_12m").sort_values("salidas_12m")
fig_top = px.bar(
    top_salidas, x="salidas_12m", y="sku_key", orientation="h",
    title="Top 15 SKU por unidades de salida registradas",
    labels={"salidas_12m": "Unidades de salida (12 meses)", "sku_key": "SKU"},
    hover_data=["meses_con_movimiento"],
    template="plotly_white"
)
fig_top.show()

### Interpretación

El ranking muestra concentración de salidas registradas: qué SKU acumulan mayor volumen durante la ventana. No equivale a los artículos más rentables, más vendidos ni más urgentes para comprar. Un SKU de alto volumen puede tener también alto inventario; para evaluar cobertura real se requiere una llave de inventario homologada y datos operativos adicionales.

## 10. Plotly 4 · treemap por marca

El treemap representa la composición del volumen de salidas por marca. Se agrega primero por marca y se evita usar categorías de clasificación de artículos, cuya semántica y cardinalidad necesitan validación.

In [ ]:
if "MARCA" in mov_12m.columns:
    marca_mov = (
        mov_12m.assign(MARCA=mov_12m["MARCA"].fillna("Sin marca"))
        .groupby("MARCA", as_index=False)
        .agg(
            salidas=("SALIDAS", "sum"),
            skus=("sku_key", "nunique")
        )
    )
    fig_tree = px.treemap(
        marca_mov,
        path=["MARCA"],
        values="salidas",
        color="salidas",
        hover_data={"skus": True, "salidas": ":,.0f"},
        title="Composición de las salidas registradas por marca",
        color_continuous_scale="Blues",
        template="plotly_white"
    )
    fig_tree.show()
else:
    print("La fuente no contiene MARCA; se omite el treemap por marca.")

### Interpretación

- El área de cada rectángulo es proporcional a las salidas acumuladas.
- El hover agrega el número de SKU distintos para contextualizar.
- Una marca grande concentra mayor volumen de salida, pero eso no indica por sí solo mayor riesgo de abastecimiento.
- Si una marca concentra muchas salidas, puede ser útil desglosar sus SKU; el treemap es una vista de composición, no de tendencia temporal.

## 11. Interactividad · filtro por marca

Usaremos `ipywidgets` para seleccionar una marca y actualizar una vista de tendencia. Si los widgets no se muestran en el entorno, se puede ejecutar la celda de alternativa estática.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    if "MARCA" not in mov_12m.columns:
        raise ValueError("No existe columna MARCA en movimientos.")

    marcas = sorted(mov_12m["MARCA"].dropna().astype(str).unique().tolist())
    selector_marca = widgets.Dropdown(
        options=["Todas"] + marcas,
        value="Todas",
        description="Marca:",
        layout=widgets.Layout(width="450px")
    )
    salida_widget = widgets.Output()

    def actualizar_tendencia(change=None):
        with salida_widget:
            clear_output(wait=True)
            df = mov_12m.copy()
            if selector_marca.value != "Todas":
                df = df[df["MARCA"].astype(str) == selector_marca.value]
            serie = df.groupby("periodo", as_index=False).agg(salidas=("SALIDAS", "sum"))
            fig = px.line(
                serie, x="periodo", y="salidas", markers=True,
                title=f"Salidas mensuales · {selector_marca.value}",
                labels={"periodo": "Mes", "salidas": "Unidades de salida"},
                template="plotly_white"
            )
            fig.update_xaxes(rangeslider_visible=True)
            fig.show()

    selector_marca.observe(actualizar_tendencia, names="value")
    display(selector_marca, salida_widget)
    actualizar_tendencia()
except Exception as e:
    print("No fue posible activar ipywidgets en este entorno:", e)
    print("Alternativa: filtra mov_12m manualmente por MARCA y vuelve a ejecutar la gráfica de tendencia.")